# Intro To Handling Dates In Pandas
Properly handling dates in Pandas can be very useful.  For example, let's say you read in a csv of temperature data with dates and you'd like to find the monthly mean temperature.  If the date column is already recognized, by pandas, as a date type, then extracting the month from the column will be very easy.  If the date column is, instead, read in as a string type, then it will be more difficult to extract the month.

## Notebook Outline
* [Using the parse dates .read_csv() to automatically read in dateformats](#parsedates)
* [Using .to_datetime() to convert a column to a pandas datetime format](#todatetime)
* [Using .dt on datetime columns](#dtattribute)
* [Introduction the .min(), .max() and .sum() methods](#minmaxsum)

In [ ]:
import pandas as pd

<a name=parsedates></a>
# Using the parse_dates argument in .read_csv() to automatically read in date formats
When reading in a file, we can specify which columns, or which combinations of columns, we would like read in as a datetime type. For this example, I am going to introduce a new data file 'LaborSheetData.csv'. This file contains real data from a very popular fast food store. Every hour, the shift manager must enter some key data in this file, like drive through times and sales for the past hour. This will be a good dataset for us to explore in some of our lectures.

First we will load the data. Remember that you will need to change the path to point to where you place the file on your computer, after you download it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
filepath = ('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/Data/LaborSheetData.csv')
laborSheetData = pd.read_csv(filepath)

#### Use the .head() method to get a look at the data, and the .info method to see the data types

In [ ]:
laborSheetData.head(2)

,Store,Business_Date,Hour,Day_Part,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,Training Store,2022-01-03,00:00,Breakfast,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,Training Store,2022-01-03,01:00,Breakfast,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00


In [ ]:
laborSheetData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Store                   42 non-null     object 
 1   Business_Date           42 non-null     object 
 2   Hour                    42 non-null     object 
 3   Day_Part                42 non-null     object 
 4   Transactions            42 non-null     int64  
 5   Net_Sales               42 non-null     float64
 6   Labor_Hours             42 non-null     float64
 7   Drive_Thru_Avg_Seconds  42 non-null     int64  
 8   Crew_Count              42 non-null     int64  
 9   Manager                 42 non-null     object 
 10  Weather                 42 non-null     object 
 11  Promotion_Flag          42 non-null     bool   
 12  Customer_Satisfaction   42 non-null     float64
 13  Timestamp               42 non-null     object 
dtypes: bool(1), float64(3), int64(3), object(7)


#### Note that data types of the Date and TimeStamp columns
The data types are object, which means they were read in as strings. Let's double check that by grabbing a value from the data column using the .loc() method and finding its type with the type function.

In [ ]:
type(laborSheetData.loc[0, 'Business_Date'])

str

#### Now, we will use the parse_dates argument to automatically read the date columns in as date types.
All we need to do is pass a list of the columns we would like pandas to try and automatically decipher as date or time objects to the parse_dates argument.  If you look above at the output from .info, you will see that the date or datetime columns are columns 2 and 13 (remember that counting starts at 0).

Note that this will cause the read_csv() method to take a little bit longer to complete.

In [ ]:
laborSheetData = pd.read_csv(filepath, parse_dates=[2, 13])

/tmp/ipykernel_2452/90362290.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  laborSheetData = pd.read_csv(filepath, parse_dates=[2, 13])


In [ ]:
laborSheetData.head(2)

,Store,Business_Date,Hour,Day_Part,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,Training Store,2022-01-03,2026-06-19 00:00:00,Breakfast,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,Training Store,2022-01-03,2026-06-19 01:00:00,Breakfast,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00


#### Now use .info() to check our datatypes.
Note that the 'Date' and 'TimeStamp' columns are now datetime types!

In [ ]:
laborSheetData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Store                   42 non-null     object        
 1   Business_Date           42 non-null     object        
 2   Hour                    42 non-null     datetime64[ns]
 3   Day_Part                42 non-null     object        
 4   Transactions            42 non-null     int64         
 5   Net_Sales               42 non-null     float64       
 6   Labor_Hours             42 non-null     float64       
 7   Drive_Thru_Avg_Seconds  42 non-null     int64         
 8   Crew_Count              42 non-null     int64         
 9   Manager                 42 non-null     object        
 10  Weather                 42 non-null     object        
 11  Promotion_Flag          42 non-null     bool          
 12  Customer_Satisfaction   42 non-null     float64     

#### Now let's use the parse_dates argument to _combine_ two (or more) columns into one datetime.
For this, I need to give you some information about this data. The 'TimeStamp' column let's us know when the data was entered, but the 'Date' + 'Hour' column let us know what hour the data is for.  So it would be great if we could combine the Date and Hour columns into one datetime column!

We can do this using the parse_dates argument. All we need to do is include a list, of the columns we want to combine, as one of the items in the list that we we pass to the parse_dates argument.  

Note that pandas will create a _new_ column called 'Date_Hour' that combines the 'Date' and 'Hour' columns two columns.

In [ ]:
laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])

/tmp/ipykernel_2452/315118610.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
/tmp/ipykernel_2452/315118610.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])


In [ ]:
laborSheetData.head(2)

,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00


In [ ]:
laborSheetData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Hour_Day_Part           42 non-null     object        
 1   Store                   42 non-null     object        
 2   Business_Date           42 non-null     object        
 3   Transactions            42 non-null     int64         
 4   Net_Sales               42 non-null     float64       
 5   Labor_Hours             42 non-null     float64       
 6   Drive_Thru_Avg_Seconds  42 non-null     int64         
 7   Crew_Count              42 non-null     int64         
 8   Manager                 42 non-null     object        
 9   Weather                 42 non-null     object        
 10  Promotion_Flag          42 non-null     bool          
 11  Customer_Satisfaction   42 non-null     float64       
 12  Timestamp               42 non-null     datetime64[n

## Let's see another example using our weather data.
Firs we will load the weather data and use .head() to look at the data.  Note that the first four columns are Year, Month, Day, and Hour.

In [ ]:
filepath = ('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/Data/724080-13739-2001')
headers = ['Year', 'Month', 'Day', 'Hour', 'Air Temp', 'Dew Point Temp', 'Sea Level Pressure',
           'Wind Direction', 'Wind Speed Rate',
           'Sky Condition Total Coverage Code',
           'Liquid Precipitation Depth Dimension - 1Hr Duration',
           'Liquid Precipitation Depth Dimension - Six Hour Duration']
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers)

/tmp/ipykernel_2452/3704144353.py:7: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  weatherData = pd.read_csv(filepath, delim_whitespace=True,


In [ ]:
weatherData.head(1)

,Year,Month,Day,Hour,Air Temp,Dew Point Temp,Sea Level Pressure,Wind Direction,Wind Speed Rate,Sky Condition Total Coverage Code,Liquid Precipitation Depth Dimension - 1Hr Duration,Liquid Precipitation Depth Dimension - Six Hour Duration
0,2022,1,1,0,80,80,10120,180,35,1,0,0


#### Now, use parse_dates to reload the data and combine the first four columns into one datetime column

In [ ]:
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers, parse_dates=[[0, 1, 2, 3]])

/tmp/ipykernel_2452/1303751619.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  weatherData = pd.read_csv(filepath, delim_whitespace=True,
/tmp/ipykernel_2452/1303751619.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  weatherData = pd.read_csv(filepath, delim_whitespace=True,
/tmp/ipykernel_2452/1303751619.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weatherData = pd.read_csv(filepath, delim_whitespace=True,


In [ ]:
weatherData.head(2)

,Year_Month_Day_Hour,Air Temp,Dew Point Temp,Sea Level Pressure,Wind Direction,Wind Speed Rate,Sky Condition Total Coverage Code,Liquid Precipitation Depth Dimension - 1Hr Duration,Liquid Precipitation Depth Dimension - Six Hour Duration
0,2022 1 1 0,80,80,10120,180,35,1,0,0
1,2022 1 1 1,85,80,10120,180,35,1,0,0


In [ ]:
weatherData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 9 columns):
 #   Column                                                    Non-Null Count  Dtype 
---  ------                                                    --------------  ----- 
 0   Year_Month_Day_Hour                                       72 non-null     object
 1   Air Temp                                                  72 non-null     int64 
 2   Dew Point Temp                                            72 non-null     int64 
 3   Sea Level Pressure                                        72 non-null     int64 
 4   Wind Direction                                            72 non-null     int64 
 5   Wind Speed Rate                                           72 non-null     int64 
 6   Sky Condition Total Coverage Code                         72 non-null     int64 
 7   Liquid Precipitation Depth Dimension - 1Hr Duration       72 non-null     int64 
 8   Liquid Precipitation Depth Dimens

<a name=todatetime></a>
# Using pandas.to_datetime() to convert a column of strings to a columns of DateTime objects
Sometimes, we need to convert a column after we read in the data. Maybe we have created the column during data processing. We can do this with the to_datetime() function in the pandas package.

I am going to read in the labor sheet data again, without using parse_dates.

In [ ]:
filepath = '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/Data/LaborSheetData.csv'
laborSheetData = pd.read_csv(filepath)
laborSheetData.head()

,Store,Business_Date,Hour,Day_Part,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,Training Store,2022-01-03,00:00,Breakfast,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,Training Store,2022-01-03,01:00,Breakfast,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00
2,Training Store,2022-01-03,02:00,Breakfast,94,817.0,12.50,138,6,Sample Manager,Clear,False,4.30,2022-01-03 02:00:00
3,Training Store,2022-01-03,03:00,Breakfast,101,900.5,13.75,147,7,Sample Manager,Clear,False,4.45,2022-01-03 03:00:00
4,Training Store,2022-01-03,04:00,Breakfast,108,984.0,15.00,156,8,Sample Manager,Rain,False,4.60,2022-01-03 04:00:00


In [ ]:
laborSheetData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Store                   42 non-null     object 
 1   Business_Date           42 non-null     object 
 2   Hour                    42 non-null     object 
 3   Day_Part                42 non-null     object 
 4   Transactions            42 non-null     int64  
 5   Net_Sales               42 non-null     float64
 6   Labor_Hours             42 non-null     float64
 7   Drive_Thru_Avg_Seconds  42 non-null     int64  
 8   Crew_Count              42 non-null     int64  
 9   Manager                 42 non-null     object 
 10  Weather                 42 non-null     object 
 11  Promotion_Flag          42 non-null     bool   
 12  Customer_Satisfaction   42 non-null     float64
 13  Timestamp               42 non-null     object 
dtypes: bool(1), float64(3), int64(3), object(7)


#### Use to_datetime to convert the 'TimeStamp' column to a column of datetime objects.
Note that to_datetime can not combine multiple columns (except under a few specific circumstances) so we can not use it to combine the 'Date' and 'Hour' columns.

In [ ]:
laborSheetData.loc[:, 'Timestamp'] = pd.to_datetime(laborSheetData['Timestamp'])

In [ ]:
laborSheetData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Store                   42 non-null     object 
 1   Business_Date           42 non-null     object 
 2   Hour                    42 non-null     object 
 3   Day_Part                42 non-null     object 
 4   Transactions            42 non-null     int64  
 5   Net_Sales               42 non-null     float64
 6   Labor_Hours             42 non-null     float64
 7   Drive_Thru_Avg_Seconds  42 non-null     int64  
 8   Crew_Count              42 non-null     int64  
 9   Manager                 42 non-null     object 
 10  Weather                 42 non-null     object 
 11  Promotion_Flag          42 non-null     bool   
 12  Customer_Satisfaction   42 non-null     float64
 13  Timestamp               42 non-null     object 
dtypes: bool(1), float64(3), int64(3), object(7)


<a name=dtattribute></a>
# Using .dt to access and operate on the datetime objects in a column
I am sure you are wondering why we went to the trouble to convert the columns to the datetime data type. Well it let's us manipulate the datetimes much more easily. Let's see some basics below:

First I read the data back in, correctly use the parse_dates argument.

In [ ]:
laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
laborSheetData.head(2)

/tmp/ipykernel_2452/3879597239.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
/tmp/ipykernel_2452/3879597239.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])


,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00


#### Use .dt to access the datetime properties, and then use .month to get the month of each value in the column

In [ ]:
laborSheetData['Timestamp'].dt.month.head(2)

,Timestamp
0,1
1,1


#### Use .dt and .year to get the year of each columns

In [ ]:
laborSheetData['Timestamp'].dt.year.head(2)

,Timestamp
0,2022
1,2022


#### Use .dt and .day to get the day of each columns

In [ ]:
laborSheetData['Timestamp'].dt.day.head(2)

,Timestamp
0,3
1,3


#### Use .dt and .hour to get the hour of each columns

In [ ]:
laborSheetData['Timestamp'].dt.hour.head(2)

,Timestamp
0,0
1,1


#### Use .dt and .hour and .value_counts() to get the row counts for each hour; do some hours get recorded less than others?

In [ ]:
laborSheetData['Timestamp'].dt.hour.value_counts()

,count
Timestamp,
0,2
1,2
2,2
3,2
4,2
5,2
6,2
7,2
8,2


#### Introducing .sort_index().  The .sort_index() method will sort the index of a dataframe (while reordering the rows appropriately)

Note that the index of the value_counts() results is made up of the values that you are counting.

In [ ]:
laborSheetData['Timestamp'].dt.hour.value_counts().sort_index()

,count
Timestamp,
0,2
1,2
2,2
3,2
4,2
5,2
6,2
7,2
8,2


<a name=minmaxsum></a>
## Introducing the .min(), .max(), and .sum() methods.
You can use these methods on any column, or dataframe to get the min, max, and sum of all numerical columns. You can also use the .min() and .max() methods to get the min and max of datetime columns

#### Use .min() and .max() to find the earliest and latest year in the data.

In [ ]:
print(laborSheetData['Timestamp'].min())
print(laborSheetData['Timestamp'].max())

2022-01-03 00:00:00
2022-01-04 17:00:00


#### Combine the use of .loc, .dt, and .sum() to find the total sales In August across all stores.

In [ ]:
laborSheetData.loc[laborSheetData['Timestamp'].dt.month == 8, 'Net_Sales'].sum()

np.float64(0.0)

#### Selecting rows greater than (or less than) a date.
Getting rows greater than, or less than (or some combination of logic tests) is fairly easy.  You use your normal logic tests: >, <, ==, etc...  and pass the value you'd like to test against as a string.  See the example below, note how we are able to use the string '2017-08-01' to get all rows with a value in the 'TimeStamp' column greater than '2017-08-01'. Easy!

In [ ]:
laborSheetData.loc[laborSheetData['Timestamp'] > '2017-08-01', :].head()

,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00
2,02:00 Breakfast,Training Store,2022-01-03,94,817.0,12.50,138,6,Sample Manager,Clear,False,4.30,2022-01-03 02:00:00
3,03:00 Breakfast,Training Store,2022-01-03,101,900.5,13.75,147,7,Sample Manager,Clear,False,4.45,2022-01-03 03:00:00
4,04:00 Breakfast,Training Store,2022-01-03,108,984.0,15.00,156,8,Sample Manager,Rain,False,4.60,2022-01-03 04:00:00


# Changing the time zone of a datetime column
Pandas includes some easy functionality to convert the timezone of a datetime column.  The first thing we need to do is 'localize' the column - this means defining what timezone the original data is in.  They data we have been using happens to be from fast food location on the west coast, so we will localize the timestamp column to the 'US/Pacific' timezone by using the tz_localize() method.

In [ ]:
laborSheetData['Timestamp'].dt.tz_localize('US/Pacific').head()

,Timestamp
0,2022-01-03 00:00:00-08:00
1,2022-01-03 01:00:00-08:00
2,2022-01-03 02:00:00-08:00
3,2022-01-03 03:00:00-08:00
4,2022-01-03 04:00:00-08:00


In [ ]:
laborSheetData.loc[:, 'Timestamp'] = laborSheetData['Timestamp'].dt.tz_localize('US/Pacific')

/tmp/ipykernel_2452/3942655177.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<DatetimeArray>
['2022-01-03 00:00:00-08:00', '2022-01-03 01:00:00-08:00',
 '2022-01-03 02:00:00-08:00', '2022-01-03 03:00:00-08:00',
 '2022-01-03 04:00:00-08:00', '2022-01-03 05:00:00-08:00',
 '2022-01-03 06:00:00-08:00', '2022-01-03 07:00:00-08:00',
 '2022-01-03 08:00:00-08:00', '2022-01-03 09:00:00-08:00',
 '2022-01-03 10:00:00-08:00', '2022-01-03 11:00:00-08:00',
 '2022-01-03 12:00:00-08:00', '2022-01-03 13:00:00-08:00',
 '2022-01-03 14:00:00-08:00', '2022-01-03 15:00:00-08:00',
 '2022-01-03 16:00:00-08:00', '2022-01-03 17:00:00-08:00',
 '2022-01-03 18:00:00-08:00', '2022-01-03 19:00:00-08:00',
 '2022-01-03 20:00:00-08:00', '2022-01-03 21:00:00-08:00',
 '2022-01-03 22:00:00-08:00', '2022-01-03 23:00:00-08:00',
 '2022-01-04 00:00:00-08:00', '2022-01-04 01:00:00-08:00',
 '2022-01-04 02:00:00-08:00', '2022-01-04 03:00:00-08:00',
 '2

In [ ]:
laborSheetData.head()

,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00-08:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00-08:00
2,02:00 Breakfast,Training Store,2022-01-03,94,817.0,12.50,138,6,Sample Manager,Clear,False,4.30,2022-01-03 02:00:00-08:00
3,03:00 Breakfast,Training Store,2022-01-03,101,900.5,13.75,147,7,Sample Manager,Clear,False,4.45,2022-01-03 03:00:00-08:00
4,04:00 Breakfast,Training Store,2022-01-03,108,984.0,15.00,156,8,Sample Manager,Rain,False,4.60,2022-01-03 04:00:00-08:00


In [ ]:
laborSheetData.loc[:, 'TimeStamp_UTC'] = laborSheetData['Timestamp'].dt.tz_convert('UTC')

In [ ]:
laborSheetData.head()

,Hour_Day_Part,Store,Business_Date,Transactions,Net_Sales,Labor_Hours,Drive_Thru_Avg_Seconds,Crew_Count,Manager,Weather,Promotion_Flag,Customer_Satisfaction,Timestamp,TimeStamp_UTC
0,00:00 Breakfast,Training Store,2022-01-03,80,650.0,10.00,120,4,Sample Manager,Rain,True,4.00,2022-01-03 00:00:00-08:00,2022-01-03 08:00:00+00:00
1,01:00 Breakfast,Training Store,2022-01-03,87,733.5,11.25,129,5,Sample Manager,Clear,False,4.15,2022-01-03 01:00:00-08:00,2022-01-03 09:00:00+00:00
2,02:00 Breakfast,Training Store,2022-01-03,94,817.0,12.50,138,6,Sample Manager,Clear,False,4.30,2022-01-03 02:00:00-08:00,2022-01-03 10:00:00+00:00
3,03:00 Breakfast,Training Store,2022-01-03,101,900.5,13.75,147,7,Sample Manager,Clear,False,4.45,2022-01-03 03:00:00-08:00,2022-01-03 11:00:00+00:00
4,04:00 Breakfast,Training Store,2022-01-03,108,984.0,15.00,156,8,Sample Manager,Rain,False,4.60,2022-01-03 04:00:00-08:00,2022-01-03 12:00:00+00:00


# Using .date_range() to create a DateTime index

We can easily create a DateTime index by using the .date_range() method. We just need to pass a start date, and end date (or a number of periods) and a frequency (the amount of time between each value in the series of values).  Possible values for freq are:
* s (for seconds)
* min (for minutes)
* H (for hours)
* D (for days)
* A (for annual, ending on the end of a year)
* AS (for annual, ending on the start of a year)
* You can also do multiples, i.e. 3H for a step of 3 hours.
* You can also use something called a DateOffset object for more custom steps, but that is beyond the scope of this course.

Note that, once you create an index, you can use it as a row index or column in a dataframe.

The docs for this method are here: <https://pandas.pydata.org/pandas-docs/stable/generated/pandas.date_range.html>

Let's look at some examples:

#### Create a DatetimeIndex with a start date of 2001-01-01, an end date of 2002-01-01 and a step of 3 hours.

In [ ]:
pd.date_range(start='2001-01-01', end='2002-01-01', freq='3H')

/tmp/ipykernel_2452/2840347437.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pd.date_range(start='2001-01-01', end='2002-01-01', freq='3H')


DatetimeIndex(['2001-01-01 00:00:00', '2001-01-01 03:00:00',
               '2001-01-01 06:00:00', '2001-01-01 09:00:00',
               '2001-01-01 12:00:00', '2001-01-01 15:00:00',
               '2001-01-01 18:00:00', '2001-01-01 21:00:00',
               '2001-01-02 00:00:00', '2001-01-02 03:00:00',
               ...
               '2001-12-30 21:00:00', '2001-12-31 00:00:00',
               '2001-12-31 03:00:00', '2001-12-31 06:00:00',
               '2001-12-31 09:00:00', '2001-12-31 12:00:00',
               '2001-12-31 15:00:00', '2001-12-31 18:00:00',
               '2001-12-31 21:00:00', '2002-01-01 00:00:00'],
              dtype='datetime64[ns]', length=2921, freq='3h')

#### Create a DatetimeIndex with a start date of 2001-01-01 and a  frequency of 1 day that continues for 100 periods.

In [ ]:
pd.date_range(start='2001-01-01', periods=100, freq='D')

DatetimeIndex(['2001-01-01', '2001-01-02', '2001-01-03', '2001-01-04',
               '2001-01-05', '2001-01-06', '2001-01-07', '2001-01-08',
               '2001-01-09', '2001-01-10', '2001-01-11', '2001-01-12',
               '2001-01-13', '2001-01-14', '2001-01-15', '2001-01-16',
               '2001-01-17', '2001-01-18', '2001-01-19', '2001-01-20',
               '2001-01-21', '2001-01-22', '2001-01-23', '2001-01-24',
               '2001-01-25', '2001-01-26', '2001-01-27', '2001-01-28',
               '2001-01-29', '2001-01-30', '2001-01-31', '2001-02-01',
               '2001-02-02', '2001-02-03', '2001-02-04', '2001-02-05',
               '2001-02-06', '2001-02-07', '2001-02-08', '2001-02-09',
               '2001-02-10', '2001-02-11', '2001-02-12', '2001-02-13',
               '2001-02-14', '2001-02-15', '2001-02-16', '2001-02-17',
               '2001-02-18', '2001-02-19', '2001-02-20', '2001-02-21',
               '2001-02-22', '2001-02-23', '2001-02-24', '2001-02-25',
      

#### Create a DatetimeIndex with a start date of 1970-01-01, that continues for 10 periods, and has an annual frequency - on the first day of each year.

In [ ]:
pd.date_range(start='1970-01-01', periods=10, freq='AS')

/tmp/ipykernel_2452/3156283219.py:1: FutureWarning: 'AS' is deprecated and will be removed in a future version, please use 'YS' instead.
  pd.date_range(start='1970-01-01', periods=10, freq='AS')


DatetimeIndex(['1970-01-01', '1971-01-01', '1972-01-01', '1973-01-01',
               '1974-01-01', '1975-01-01', '1976-01-01', '1977-01-01',
               '1978-01-01', '1979-01-01'],
              dtype='datetime64[ns]', freq='YS-JAN')

#### Create a DatetimeIndex with a start date of 1970-01-01, that continues for 10 periods, and has an annual frequency - on the last day of each year.

In [ ]:
pd.date_range(start='1969-12-31', periods=10, freq='A')

/tmp/ipykernel_2452/2341819708.py:1: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  pd.date_range(start='1969-12-31', periods=10, freq='A')


DatetimeIndex(['1969-12-31', '1970-12-31', '1971-12-31', '1972-12-31',
               '1973-12-31', '1974-12-31', '1975-12-31', '1976-12-31',
               '1977-12-31', '1978-12-31'],
              dtype='datetime64[ns]', freq='YE-DEC')

#### Create a DatetimeIndex with a start date of 2001-01-01 and a step size of 1 day that continues for 100 steps.

In [ ]:
pd.date_range(start='1969-12-31', periods=10, freq='A-AUG')

/tmp/ipykernel_2452/2874970740.py:1: FutureWarning: 'A-AUG' is deprecated and will be removed in a future version, please use 'YE-AUG' instead.
  pd.date_range(start='1969-12-31', periods=10, freq='A-AUG')


DatetimeIndex(['1970-08-31', '1971-08-31', '1972-08-31', '1973-08-31',
               '1974-08-31', '1975-08-31', '1976-08-31', '1977-08-31',
               '1978-08-31', '1979-08-31'],
              dtype='datetime64[ns]', freq='YE-AUG')


---

# Additional students' Practice and Answer Key

These practice questions are designed for undergraduate students. Try the **Student Practice** cell first, then compare your work with the **Completed Answer** cell.


## Student Practice 1: Convert Dates
Create a DataFrame with a text date column and convert it to datetime format.

## Student Practice 2: Extract Month
Create a new column that stores the month from the date.

In [ ]:
# Completed Answer: Date Practice
import pandas as pd

df = pd.DataFrame({
    'order_date': ['2026-01-15', '2026-02-20', '2026-03-05'],
    'sales': [120, 150, 90]
})

df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.month

display(df)